# 04 – Predictive Modeling

Two machine learning tasks:

1. **Win Prediction** — Classify whether the home team wins a given game  
   Models: Logistic Regression, Random Forest  
   Metrics: Accuracy, ROC-AUC, Confusion Matrix

2. **Player Scoring Regression** — Predict a player's points-per-game season average  
   Model: Ridge Regression  
   Metrics: RMSE, R²

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

from src.data_loader import load_games, load_games_details
from src.features import (
    add_rolling_team_stats, add_true_shooting, add_usage_rate,
    aggregate_player_season_stats,
)
from src.models import (
    build_win_prediction_dataset, train_logistic_regression, train_random_forest,
    evaluate_classifier, run_cv_report,
    build_scoring_dataset, train_ridge_regression, evaluate_regressor,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

RAW_DIR = os.path.join(os.path.abspath('..'), 'data', 'raw')
PROCESSED_DIR = os.path.join(os.path.abspath('..'), 'data', 'processed')

## Part A — Win Prediction

### A1. Build Feature Dataset

In [ ]:
games_feat_path = os.path.join(PROCESSED_DIR, 'games_features.csv')
if os.path.exists(games_feat_path):
    games = pd.read_csv(games_feat_path, parse_dates=['GAME_DATE_EST'])
else:
    games = load_games(RAW_DIR)
    games = add_rolling_team_stats(games, window=10)

X_win, y_win = build_win_prediction_dataset(games)
print('Samples:', len(X_win), '| Class balance:', y_win.value_counts().to_dict())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_win, y_win, test_size=0.2, random_state=42, stratify=y_win
)

### A2. Logistic Regression

In [ ]:
lr_model = train_logistic_regression(X_train, y_train)
lr_metrics = evaluate_classifier(lr_model, X_test, y_test)

print(f"Accuracy : {lr_metrics['accuracy']:.4f}")
print(f"ROC-AUC  : {lr_metrics.get('roc_auc', 'N/A')}")
print()
print(lr_metrics['classification_report'])

### A3. Random Forest

In [ ]:
rf_model = train_random_forest(X_train, y_train)
rf_metrics = evaluate_classifier(rf_model, X_test, y_test)

print(f"Accuracy : {rf_metrics['accuracy']:.4f}")
print(f"ROC-AUC  : {rf_metrics.get('roc_auc', 'N/A')}")
print()
print(rf_metrics['classification_report'])

### A4. Cross-Validation

In [ ]:
cv_rf = run_cv_report(rf_model, X_win, y_win, cv=5, scoring='accuracy')
print(f"RF 5-fold CV accuracy: {cv_rf['cv_mean']:.4f} ± {cv_rf['cv_std']:.4f}")

### A5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(rf_metrics['confusion_matrix'], display_labels=['Away Win', 'Home Win']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Random Forest — Confusion Matrix')

RocCurveDisplay.from_estimator(rf_model, X_test, y_test, ax=axes[1], name='Random Forest')
RocCurveDisplay.from_estimator(lr_model, X_test, y_test, ax=axes[1], name='Logistic Regression')
axes[1].set_title('ROC Curve Comparison')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importances (Random Forest)
importances = pd.Series(rf_model.feature_importances_, index=X_win.columns).sort_values()

plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

## Part B — Player Scoring Regression

### B1. Build Feature Dataset

In [ ]:
season_path = os.path.join(PROCESSED_DIR, 'player_season_stats.csv')
if os.path.exists(season_path):
    player_season = pd.read_csv(season_path)
else:
    details = load_games_details(RAW_DIR)
    if 'SEASON' in games.columns:
        season_map = games.set_index('GAME_ID')['SEASON']
        details['SEASON'] = details['GAME_ID'].map(season_map)
    details = add_true_shooting(details)
    details = add_usage_rate(details)
    player_season = aggregate_player_season_stats(details)

X_score, y_score = build_scoring_dataset(player_season)
print('Samples:', len(X_score))

### B2. Train Ridge Regression

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_score, y_score, test_size=0.2, random_state=42)

ridge_model = train_ridge_regression(X_tr, y_tr, alpha=1.0)
reg_metrics = evaluate_regressor(ridge_model, X_te, y_te)

print(f"RMSE : {reg_metrics['rmse']:.4f}")
print(f"R²   : {reg_metrics['r2']:.4f}")

### B3. Predicted vs Actual Plot

In [ ]:
y_pred = ridge_model.predict(X_te)

plt.figure(figsize=(7, 6))
plt.scatter(y_te, y_pred, alpha=0.4, color='royalblue', edgecolors='none')
lims = [min(y_te.min(), y_pred.min()) - 1, max(y_te.max(), y_pred.max()) + 1]
plt.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual PTS/G')
plt.ylabel('Predicted PTS/G')
plt.title(f'Ridge Regression — Predicted vs Actual (R²={reg_metrics["r2"]:.3f})')
plt.legend()
plt.tight_layout()
plt.show()